# Experiment 1: "How Many Steps Earlier Does an Object Appear?"
### Measuring the Early Emergence Probability of Hallucinated vs. Real Objects in LLaVA-1.5-7B

---

## Execution Roadmap (Gate Milestones)
1. **Environment Configuration and GPU Verification**
2. **Codebase Acquisition from GitHub and Dependency Setup**
3. **COCO 2014 Data Preparation (Annotations and Validation Images with tqdm Progress Tracking)**
4. **Gate G0: Automated Unit Testing (T1-T3)**
5. **Gate G1: Smoke Testing (3 Images) and GPU Sanity Verification (T4-T6)**
6. **Gate G2: Pilot Execution (100 Development Images)**
7. **Gate G3: Development Cohort Execution (500 Images) and CHAIR Sanity Check**
8. **Gate G4: Full Development Analysis (Questions A and B, Visualizations, and Draft Report)**
9. **Gate G5: Confirmation Cohort Execution (2,000 Images) and Final Report Generation**
10. **Results Export (Download Artifacts Archive)**

> Note: This workflow strictly adheres to the Gate criteria specified in `EXPERIMENT1_SPEC.md`. Each gate must successfully pass before proceeding to subsequent stages.

## 1. Environment Configuration and GPU Verification
Inspect GPU availability and allocated video memory (VRAM).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Codebase Acquisition from GitHub and Dependency Setup
Clones the repository and installs the required packages.

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/ntmy12/Experiment_CTI.git"
REPO_DIR = "/kaggle/working/Experiment_CTI"

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    print(f"Repository already present at {REPO_DIR}")

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q bitsandbytes

## 3. COCO 2014 Data Preparation
Downloads official annotations (`instances_val2014.json`, `captions_val2014.json`) and validation images with a real-time progress bar.

In [ ]:
import os
import glob
import urllib.request
from tqdm import tqdm

%cd {REPO_DIR}
os.makedirs("data", exist_ok=True)

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url, out_path):
    print(f"Downloading {os.path.basename(out_path)}...")
    with DownloadProgressBar(unit="B", unit_scale=True, miniters=1, desc=os.path.basename(out_path), dynamic_ncols=True) as t:
        urllib.request.urlretrieve(url, filename=out_path, reporthook=t.update_to)

# 1. Link or download COCO images (val2014)
candidate_val_dirs = [
    "/kaggle/input/datasets/biminhco/val2014",
    "/kaggle/input/val2014",
] + glob.glob("/kaggle/input/**/val2014", recursive=True)

matched_val_dir = None
for path in candidate_val_dirs:
    if os.path.exists(path) and os.path.isdir(path):
        matched_val_dir = path
        break

if not os.path.exists("data/val2014"):
    if matched_val_dir:
        print(f"Found COCO val2014 in Kaggle dataset at: {matched_val_dir}")
        os.symlink(matched_val_dir, "data/val2014")
        print("Successfully created symlink: data/val2014 -> " + matched_val_dir)
    else:
        print("No local Kaggle val2014 dataset found. Downloading from official COCO source (6.2 GB)...")
        download_file("http://images.cocodataset.org/zips/val2014.zip", "data/val2014.zip")
        print("Extracting val2014.zip...")
        !unzip -q data/val2014.zip -d data/
        !rm -f data/val2014.zip
        print("val2014 images extracted to data/val2014/")
else:
    print("data/val2014 already configured.")

# 2. Link or download COCO Annotations (~20 MB)
candidate_instances = glob.glob("/kaggle/input/**/instances_val2014.json", recursive=True)
candidate_captions = glob.glob("/kaggle/input/**/captions_val2014.json", recursive=True)

if not os.path.exists("data/instances_val2014.json") or not os.path.exists("data/captions_val2014.json"):
    if candidate_instances and candidate_captions:
        os.symlink(candidate_instances[0], "data/instances_val2014.json")
        os.symlink(candidate_captions[0], "data/captions_val2014.json")
        print(f"Linked annotations from Kaggle dataset: {candidate_instances[0]}")
    else:
        print("Downloading COCO annotations (~20 MB)...")
        download_file("http://images.cocodataset.org/annotations/annotations_trainval2014.zip", "data/annotations.zip")
        !unzip -q -o data/annotations.zip annotations/instances_val2014.json annotations/captions_val2014.json -d data/
        !mv data/annotations/instances_val2014.json data/instances_val2014.json
        !mv data/annotations/captions_val2014.json data/captions_val2014.json
        !rm -rf data/annotations data/annotations.zip
        print("Annotations extracted successfully to data/")
else:
    print("Annotations already exist in data/")


## 4. Gate G0: Automated Unit Testing (T1-T3)
**Pass Criteria:** Tests T1 (token piece alignment, text reconstruction, mention extraction), T2 (1:1 control matching, caliper constraint, seed determinism), and T3 (Bootstrap CI, Holm-Bonferroni correction, AUROC) must achieve a 100% pass rate.

In [ ]:
%cd {REPO_DIR}
!python3 -m unittest discover -s tests -p "test_common.py" -v

## 5. Gate G1: Smoke Testing (3 Images) and GPU Sanity (T4-T6)
**Objective:** Verify pipeline execution across 3 test images and validate position alignment accuracy (`argmax(pred[i]) == gen_ids[i]` >= 98%) to detect off-by-one errors.

In [ ]:
%cd {REPO_DIR}

# 1. Generate captions for 3 smoke images
!python3 src/01_generate.py --split smoke --n_images 3 --output_file data/smoke_captions.jsonl

# 2. Perform CHAIR labeling
!python3 src/02_label_chair.py --captions_file data/smoke_captions.jsonl --output_file data/smoke_labels.jsonl

# Inspect smoke sample
import json
with open("data/smoke_labels.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        print(f"Image ID {item['image_id']}: {len(item['mentions'])} mentions")
        for m in item['mentions']:
            print(f"   - {m['word']} (canon: {m['canon']}, tok: {m['tok_idx']}, halluc: {m['halluc']})")

# 3. Execute GPU alignment sanity test T4
!python3 -m unittest tests/test_gpu.py -v

## 6. Gate G2: Pilot Execution (100 Development Images)
**Objective:** Execute the entire end-to-end pipeline on 100 development images, confirming all output schemas and verifying data attrition in `funnel.json`.

In [ ]:
%cd {REPO_DIR}

# Generate captions for 100 dev images
!python3 src/01_generate.py --split dev --offset 0 --n_images 100 --output_file data/captions.jsonl

# CHAIR labeling
!python3 src/02_label_chair.py --captions_file data/captions.jsonl --output_file data/labels.jsonl

# Question A: Lag curve analysis
!python3 src/03_lag_curve.py --labels_file data/labels.jsonl --output_dir results/exp1_pilot

# Question B: PMC analysis
!python3 src/04_pmc.py --labels_file data/labels.jsonl --captions_file data/captions.jsonl --output_dir results/exp1_pilot

# Visualization and pilot report
!python3 src/05_visualize_report.py --captions_file data/captions.jsonl --labels_file data/labels.jsonl --output_dir results/exp1_pilot

# Display pilot funnel
with open("results/exp1_pilot/funnel.json") as f:
    print(json.dumps(json.load(f), indent=2))

## 7. Gate G3: Development Cohort Execution (500 Images) and CHAIR Sanity
Processes the full 500-image development cohort (`[0, 500)`). Evaluates CHAIR sanity benchmarks (referencing literature values: CHAIR_I ~ 6.0%, CHAIR_S ~ 19.6%).

In [ ]:
%cd {REPO_DIR}

# Generate captions for remaining dev cohort images (resumes automatically)
!python3 src/01_generate.py --split dev --offset 0 --n_images 500 --output_file data/captions.jsonl

# CHAIR labeling across 500 dev images
!python3 src/02_label_chair.py --captions_file data/captions.jsonl --output_file data/labels.jsonl

## 8. Gate G4: Full Development Analysis (Questions A and B, Figures, and Draft Report)
Executes batched teacher-forcing forward evaluation, paired statistical testing, PMC extraction, figure rendering, and draft report generation.

In [ ]:
%cd {REPO_DIR}

# 1. Question A: Lag curve analysis
!python3 src/03_lag_curve.py --labels_file data/labels.jsonl --output_dir results/exp1

# 2. Question B: PMC analysis
!python3 src/04_pmc.py --labels_file data/labels.jsonl --captions_file data/captions.jsonl --output_dir results/exp1

# 3. Render figures and compile REPORT.md
!python3 src/05_visualize_report.py --captions_file data/captions.jsonl --labels_file data/labels.jsonl --output_dir results/exp1

In [ ]:
# Display generated visual figures
from IPython.display import Image, display
import os

figs = [
    "results/exp1/figures/s_vs_m.png",
    "results/exp1/figures/delta_vs_m.png",
    "results/exp1/figures/auroc_vs_m.png",
    "results/exp1/figures/pmc_by_group.png",
    "results/exp1/figures/pmc_by_nprec.png",
]

for f in figs:
    if os.path.exists(f):
        print(f"=== {f} ===")
        display(Image(filename=f))

# Display draft REPORT.md summary
if os.path.exists("results/exp1/REPORT.md"):
    with open("results/exp1/REPORT.md") as r_file:
        print(r_file.read()[:2000])

## 9. Gate G5: Confirmation Cohort Execution (2,000 Images) and Final Report Generation
> Caution: Gate G5 must only be executed after G0-G4 pass criteria are verified and parameters are frozen.
Runs independent confirmation analysis on the non-overlapping cohort of 2,000 images (`[500, 2500)`).

In [ ]:
%cd {REPO_DIR}

# Generate captions for confirmation split [500, 2500)
!python3 src/01_generate.py --split confirm --offset 500 --n_images 2000 --output_file data/captions_confirm.jsonl

# CHAIR labeling for confirmation cohort
!python3 src/02_label_chair.py --captions_file data/captions_confirm.jsonl --output_file data/labels_confirm.jsonl

# Question A analysis on confirmation split
!python3 src/03_lag_curve.py --labels_file data/labels_confirm.jsonl --output_dir results/exp1_confirm

# Question B analysis on confirmation split
!python3 src/04_pmc.py --labels_file data/labels_confirm.jsonl --captions_file data/captions_confirm.jsonl --output_dir results/exp1_confirm

# Final figures and REPORT.md compilation
!python3 src/05_visualize_report.py --captions_file data/captions_confirm.jsonl --labels_file data/labels_confirm.jsonl --output_dir results/exp1_confirm

## 10. Results Export (Download Artifacts Archive)
Compresses all output tables, figures, manifest, and reports into a zip archive for export.

In [ ]:
%cd {REPO_DIR}
!zip -r /kaggle/working/experiment1_results.zip results/
print("Experiment artifacts successfully archived to: /kaggle/working/experiment1_results.zip")
print("Download the archive directly via the 'Output' tab in the Kaggle sidebar.")